# Initial-methionine retained-versus-removed sensitivity analysis

This notebook evaluates whether optional removal of the initial methionine changes
the identities, rankings, or sulfur-containing amino-acid (S-AA) composition of
the ten highest-S-AA proteins selected for each crop proteome.

The analysis compares two preprocessing conditions:

- `retained`: the normalized original sequence retains its initial methionine;
- `removed`: an initial methionine is removed when it is the first residue of the
  normalized original sequence, before optional canonical-residue filtering.

The input workbook combines protein-level composition descriptors and UniProt
annotations for four species under both conditions. The notebook produces:

1. top-10 protein-set overlap and Jaccard similarity;
2. rank and composition changes for proteins shared between conditions; and
3. species-level sensitivity summaries.

All percentage-point changes are calculated as **removed minus retained**.

## 1. Imports, paths, and reproducibility information

Run this notebook from the repository root, where the input workbook is stored.
The input and output filenames are defined explicitly below to avoid accidental
use of similarly named working files.

In [3]:
from pathlib import Path
import platform
import sys

import pandas as pd


PROJECT_ROOT = Path.cwd().resolve()

# INPUT_FILE = (
#     PROJECT_ROOT
#     / "Cross_species_top10_SAA_descriptors_annotations.xlsx"
# )

INPUT_FILE = PROJECT_ROOT / "data" / "Cross_species_top10_compact_analysis_table.xlsx"

OUTPUT_FILE = (
    PROJECT_ROOT
    / "Initial_Met_sensitivity_analysis.xlsx"
)

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        "The required input workbook was not found. "
        f"Expected location: {INPUT_FILE}"
    )

print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())
print("pandas version:", pd.__version__)
print("Project root:", PROJECT_ROOT)
print("Input file:", INPUT_FILE.name)
print("Output file:", OUTPUT_FILE.name)

Python version: 3.13.15
Platform: Windows-10-10.0.19045-SP0
pandas version: 3.0.1
Project root: G:\Other computers\My Laptop\Desktop\Proteome_analysis_S_AA
Input file: Cross_species_top10_compact_analysis_table.xlsx
Output file: Initial_Met_sensitivity_analysis.xlsx


## 2. Read and validate the compact analysis table

The workflow requires one row per species, preprocessing condition, and protein.
Each species-condition group must contain ten unique protein identifiers. These
checks prevent duplicated rows, missing conditions, or incomplete top-10 lists
from silently altering the overlap statistics.

In [4]:
df = pd.read_excel(INPUT_FILE)

required_columns = {
    "species",
    "initial_met_condition",
    "protein_id",
    "protein_name",
    "saa_rank",
    "met_pct",
    "cys_pct",
    "saa_pct",
}

missing_columns = sorted(required_columns - set(df.columns))
if missing_columns:
    raise ValueError(
        "The input workbook is missing required columns: "
        + ", ".join(missing_columns)
    )

expected_conditions = {"removed", "retained"}
observed_conditions = set(
    df["initial_met_condition"].dropna().astype(str)
)

if observed_conditions != expected_conditions:
    raise ValueError(
        "Expected initial_met_condition values "
        f"{sorted(expected_conditions)}, but found "
        f"{sorted(observed_conditions)}."
    )

duplicate_keys = df.duplicated(
    subset=["species", "initial_met_condition", "protein_id"],
    keep=False,
)

if duplicate_keys.any():
    duplicated_rows = df.loc[
        duplicate_keys,
        ["species", "initial_met_condition", "protein_id"],
    ]
    raise ValueError(
        "Duplicate species-condition-protein rows were detected:\n"
        + duplicated_rows.to_string(index=False)
    )

group_counts = (
    df.groupby(["species", "initial_met_condition"])["protein_id"]
    .nunique()
    .rename("n_unique_proteins")
    .reset_index()
)

invalid_groups = group_counts[
    group_counts["n_unique_proteins"] != 10
]

if not invalid_groups.empty:
    raise ValueError(
        "Each species-condition group must contain exactly 10 "
        "unique proteins:\n"
        + invalid_groups.to_string(index=False)
    )

print("Input dimensions:", df.shape)
print("Species:", df["species"].nunique())
print("Validated species-condition groups:", len(group_counts))

df.head()

Input dimensions: (80, 46)
Species: 4
Validated species-condition groups: 8


,species,initial_met_condition,saa_rank,protein_id,protein_name,aa_count_raw,aa_count_adjusted,remove_start_m,start_m_removed,sequence_status,...,aa_M_freq,aa_N_freq,aa_P_freq,aa_Q_freq,aa_R_freq,aa_S_freq,aa_T_freq,aa_V_freq,aa_W_freq,aa_Y_freq
0,Cicer arietinum,removed,1,A0A1S2XNX3,Metallothionein-like protein 4B,79,78,True,True,Complete,...,0.012821,0.064103,0.038462,0.000000,0.064103,0.102564,0.128205,0.038462,0.000000,0.000000
1,Cicer arietinum,removed,2,A0A3Q7K730,Metallothionein-like protein,79,78,True,True,Complete,...,0.051282,0.051282,0.025641,0.025641,0.000000,0.102564,0.089744,0.025641,0.000000,0.025641
2,Cicer arietinum,removed,3,Q39459,Metallothionein-like protein 2,79,78,True,True,Complete,...,0.051282,0.051282,0.025641,0.025641,0.000000,0.102564,0.089744,0.025641,0.000000,0.025641
3,Cicer arietinum,removed,4,A0A1S2YPQ3,G protein gamma domain-containing protein,224,223,True,True,Complete,...,0.013453,0.040359,0.116592,0.008969,0.044843,0.152466,0.026906,0.022422,0.017937,0.008969
4,Cicer arietinum,removed,5,A0A1S2YUR7,Class II metallothionein-like protein 1A,78,77,True,True,Complete,...,0.000000,0.038961,0.038961,0.000000,0.038961,0.103896,0.129870,0.051948,0.000000,0.000000


## 3. Top-10 overlap and Jaccard similarity

For each species, the retained and removed top-10 protein sets are compared.

- `n_shared` is the number of proteins occurring in both lists.
- `overlap_pct` is the shared count divided by the smaller list size. Because
  validation requires ten proteins in each list, this is equivalent to
  `n_shared / 10 × 100`.
- `jaccard_similarity` is the intersection size divided by the union size.
- Condition-specific protein identifiers are retained to document substitutions
  between the two top-10 lists.

In [5]:
overlap_results = []

for species, df_species in df.groupby("species", sort=True):

    removed_ids = set(
        df_species.loc[
            df_species["initial_met_condition"] == "removed",
            "protein_id",
        ]
    )

    retained_ids = set(
        df_species.loc[
            df_species["initial_met_condition"] == "retained",
            "protein_id",
        ]
    )

    shared_ids = removed_ids & retained_ids
    union_ids = removed_ids | retained_ids
    removed_only = removed_ids - retained_ids
    retained_only = retained_ids - removed_ids

    n_removed = len(removed_ids)
    n_retained = len(retained_ids)
    n_shared = len(shared_ids)
    n_union = len(union_ids)

    overlap_results.append(
        {
            "species": species,
            "n_removed": n_removed,
            "n_retained": n_retained,
            "n_shared": n_shared,
            "overlap_pct": (
                n_shared / min(n_removed, n_retained) * 100
            ),
            "n_union": n_union,
            "jaccard_similarity": n_shared / n_union,
            "removed_only_proteins": ", ".join(
                sorted(removed_only)
            ),
            "retained_only_proteins": ", ".join(
                sorted(retained_only)
            ),
        }
    )

df_overlap = pd.DataFrame(overlap_results)

df_overlap

,species,n_removed,n_retained,n_shared,overlap_pct,n_union,jaccard_similarity,removed_only_proteins,retained_only_proteins
0,Cicer arietinum,10,10,9,90.0,11,0.818182,A0A1S2XRI4,Q39458
1,Glycine max,10,10,9,90.0,11,0.818182,K7LC96,P01063
2,Lupinus albus,10,10,9,90.0,11,0.818182,A0A6A4Q7Q1,A0A6A4NMY5
3,Phaseolus vulgaris,10,10,9,90.0,11,0.818182,V7BSW9,V7ARR1


## 4. Match proteins shared between preprocessing conditions

The retained and removed tables are renamed before merging so that both sets of
rank and composition values remain traceable. Only proteins present in both
top-10 lists are included in the paired comparison.

Change variables are interpreted as follows:

- `rank_change = removed rank − retained rank`; a positive value indicates a
  numerically lower position after initial-methionine removal, whereas a negative
  value indicates a numerically higher position.
- `delta_met_pct`, `delta_cys_pct`, and `delta_saa_pct` are percentage-point
  differences calculated as removed minus retained.

In [6]:
df_removed = (
    df.loc[df["initial_met_condition"] == "removed"]
    .copy()
    .rename(
        columns={
            "saa_rank": "saa_rank_removed",
            "met_pct": "met_pct_removed",
            "cys_pct": "cys_pct_removed",
            "saa_pct": "saa_pct_removed",
        }
    )
)

df_retained = (
    df.loc[df["initial_met_condition"] == "retained"]
    .copy()
    .rename(
        columns={
            "saa_rank": "saa_rank_retained",
            "met_pct": "met_pct_retained",
            "cys_pct": "cys_pct_retained",
            "saa_pct": "saa_pct_retained",
        }
    )
)

comparison_columns_removed = [
    "species",
    "protein_id",
    "protein_name",
    "saa_rank_removed",
    "met_pct_removed",
    "cys_pct_removed",
    "saa_pct_removed",
]

comparison_columns_retained = [
    "species",
    "protein_id",
    "saa_rank_retained",
    "met_pct_retained",
    "cys_pct_retained",
    "saa_pct_retained",
]

df_shared = pd.merge(
    df_removed[comparison_columns_removed],
    df_retained[comparison_columns_retained],
    on=["species", "protein_id"],
    how="inner",
    validate="one_to_one",
)

df_shared["rank_change"] = (
    df_shared["saa_rank_removed"]
    - df_shared["saa_rank_retained"]
)

df_shared["absolute_rank_change"] = (
    df_shared["rank_change"].abs()
)

df_shared["delta_met_pct"] = (
    df_shared["met_pct_removed"]
    - df_shared["met_pct_retained"]
)

df_shared["delta_cys_pct"] = (
    df_shared["cys_pct_removed"]
    - df_shared["cys_pct_retained"]
)

df_shared["delta_saa_pct"] = (
    df_shared["saa_pct_removed"]
    - df_shared["saa_pct_retained"]
)

df_shared = (
    df_shared
    .sort_values(["species", "saa_rank_retained"])
    .reset_index(drop=True)
)

# Verify that the paired rows agree with the independently
# calculated overlap counts.
paired_counts = (
    df_shared.groupby("species")["protein_id"]
    .size()
    .rename("paired_n_shared")
    .reset_index()
)

overlap_check = df_overlap.merge(
    paired_counts,
    on="species",
    how="left",
    validate="one_to_one",
)

if not (
    overlap_check["n_shared"]
    == overlap_check["paired_n_shared"]
).all():
    raise RuntimeError(
        "The paired comparison does not match the overlap counts."
    )

df_shared.head(20)

,species,protein_id,protein_name,saa_rank_removed,met_pct_removed,cys_pct_removed,saa_pct_removed,saa_rank_retained,met_pct_retained,cys_pct_retained,saa_pct_retained,rank_change,absolute_rank_change,delta_met_pct,delta_cys_pct,delta_saa_pct
0,Cicer arietinum,A0A1S2XNX3,Metallothionein-like protein 4B,1,1.282051,21.794872,23.076923,1,2.531646,21.518987,24.050633,0,0,-1.249594,0.275884,-0.973710
1,Cicer arietinum,A0A3Q7K730,Metallothionein-like protein,2,5.128205,17.948718,23.076923,2,6.329114,17.721519,24.050633,0,0,-1.200909,0.227199,-0.973710
2,Cicer arietinum,Q39459,Metallothionein-like protein 2,3,5.128205,17.948718,23.076923,3,6.329114,17.721519,24.050633,0,0,-1.200909,0.227199,-0.973710
3,Cicer arietinum,A0A1S2YUR7,Class II metallothionein-like protein 1A,5,0.000000,22.077922,22.077922,4,1.282051,21.794872,23.076923,1,1,-1.282051,0.283050,-0.999001
4,Cicer arietinum,A0A1S2YPQ3,G protein gamma domain-containing protein,4,1.345291,21.076233,22.421525,5,1.785714,20.982143,22.767857,-1,1,-0.440423,0.094090,-0.346332
5,Cicer arietinum,A0A1S2XEG2,G protein gamma domain-containing protein,6,2.912621,18.932039,21.844660,6,3.381643,18.840580,22.222222,0,0,-0.469021,0.091459,-0.377562
6,Cicer arietinum,A0A1S2Y589,Peamaclen-like,7,4.545455,14.772727,19.318182,7,5.617978,14.606742,20.224719,0,0,-1.072523,0.165986,-0.906537
7,Cicer arietinum,A0A1S3E7N8,Uncharacterized protein,8,3.816794,14.503817,18.320611,8,4.545455,14.393939,18.939394,0,0,-0.728661,0.109877,-0.618783
8,Cicer arietinum,A0A3Q7K738,Metallothionein-like protein,10,1.351351,16.216216,17.567568,9,2.666667,16.000000,18.666667,1,1,-1.315315,0.216216,-1.099099
9,Glycine max,A0A0R0G5L9,Cysteine-rich transmembrane domain-containing ...,1,3.636364,23.636364,27.272727,1,5.357143,23.214286,28.571429,0,0,-1.720779,0.422078,-1.298701


## 5. Species-level sensitivity summary

The summary quantifies list stability, rank stability, and composition changes
among shared proteins. Mean signed deltas indicate the direction of change,
whereas maximum absolute deltas describe the largest observed change regardless
of direction.

In [7]:
df_sensitivity_summary = (
    df_shared
    .groupby("species")
    .agg(
        n_shared=("protein_id", "size"),
        mean_absolute_rank_change=(
            "absolute_rank_change",
            "mean",
        ),
        maximum_absolute_rank_change=(
            "absolute_rank_change",
            "max",
        ),
        unchanged_rank_count=(
            "absolute_rank_change",
            lambda x: (x == 0).sum(),
        ),
        mean_delta_met_pct=(
            "delta_met_pct",
            "mean",
        ),
        maximum_absolute_delta_met_pct=(
            "delta_met_pct",
            lambda x: x.abs().max(),
        ),
        mean_delta_cys_pct=(
            "delta_cys_pct",
            "mean",
        ),
        maximum_absolute_delta_cys_pct=(
            "delta_cys_pct",
            lambda x: x.abs().max(),
        ),
        mean_delta_saa_pct=(
            "delta_saa_pct",
            "mean",
        ),
        maximum_absolute_delta_saa_pct=(
            "delta_saa_pct",
            lambda x: x.abs().max(),
        ),
    )
    .reset_index()
)

df_sensitivity_summary = pd.merge(
    df_overlap,
    df_sensitivity_summary,
    on=["species", "n_shared"],
    how="left",
    validate="one_to_one",
)

round_columns = [
    "overlap_pct",
    "jaccard_similarity",
    "mean_absolute_rank_change",
    "maximum_absolute_rank_change",
    "mean_delta_met_pct",
    "maximum_absolute_delta_met_pct",
    "mean_delta_cys_pct",
    "maximum_absolute_delta_cys_pct",
    "mean_delta_saa_pct",
    "maximum_absolute_delta_saa_pct",
]

df_sensitivity_summary[round_columns] = (
    df_sensitivity_summary[round_columns]
    .round(4)
)

df_sensitivity_summary

,species,n_removed,n_retained,n_shared,overlap_pct,n_union,jaccard_similarity,removed_only_proteins,retained_only_proteins,mean_absolute_rank_change,maximum_absolute_rank_change,unchanged_rank_count,mean_delta_met_pct,maximum_absolute_delta_met_pct,mean_delta_cys_pct,maximum_absolute_delta_cys_pct,mean_delta_saa_pct,maximum_absolute_delta_saa_pct
0,Cicer arietinum,10,10,9,90.0,11,0.8182,A0A1S2XRI4,Q39458,0.3333,1,6,-0.9955,1.3153,0.1879,0.2831,-0.8076,1.0991
1,Glycine max,10,10,9,90.0,11,0.8182,K7LC96,P01063,1.1111,2,2,-1.2302,1.7532,0.2478,0.4221,-0.9825,1.3471
2,Lupinus albus,10,10,9,90.0,11,0.8182,A0A6A4Q7Q1,A0A6A4NMY5,0.1111,1,8,-1.0767,1.5574,0.1806,0.3005,-0.8962,1.2568
3,Phaseolus vulgaris,10,10,9,90.0,11,0.8182,V7BSW9,V7ARR1,1.2222,3,3,-1.3408,3.6667,0.2501,0.5000,-1.0907,3.1667


## 6. Export the sensitivity-analysis workbook

The workbook contains three sheets:

- `Protein_overlap`: overlap statistics and condition-specific substitutions;
- `Shared_protein_changes`: paired rank and composition changes for shared
  proteins;
- `Species_summary`: species-level stability and sensitivity statistics.

The output file is overwritten only when this export cell is executed.

In [8]:
with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl",
) as writer:

    df_overlap.to_excel(
        writer,
        sheet_name="Protein_overlap",
        index=False,
    )

    df_shared.to_excel(
        writer,
        sheet_name="Shared_protein_changes",
        index=False,
    )

    df_sensitivity_summary.to_excel(
        writer,
        sheet_name="Species_summary",
        index=False,
    )

print(f"Results saved to:\n{OUTPUT_FILE}")

Results saved to:
G:\Other computers\My Laptop\Desktop\Proteome_analysis_S_AA\Initial_Met_sensitivity_analysis.xlsx


## Reproducibility note

Before committing the notebook to GitHub, restart the kernel, clear all outputs,
run all cells sequentially, and save the executed notebook. The input workbook
and exported sensitivity workbook should remain in the repository root unless
the paths in this notebook are updated consistently.